# Stage 8: Evaluation Ground Truth Generation (Colab + Ollama)

This notebook handles the generation of the 50-query Ground Truth dataset while entirely avoiding cloud API rate limits. It is explicitly designed to run on a **Google Colab GPU instance** using the local Ollama runtime to run the highly capable Phi-4 Mini model for advanced reasoning.

### Setup Instructions
1. **Hardware Setup**: A GPU runtime in Colab is required (`Runtime -> Change runtime type -> T4 GPU`).
2. **Data Mounting**: Google Drive must be mounted to access the `sd-land-use-rag` project folder.

In [ ]:
# Mount Google Drive if running in Colab to access data files
import sys
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # IMPORTANT: Adjust this path to the correct project directory in Google Drive
    base_path = '/content/drive/MyDrive/sd-land-use-rag'
else:
    # Fallback for local execution
    base_path = '..'

In [ ]:
# Install and start the Ollama Daemon natively inside Colab
if 'google.colab' in sys.modules:
    # 1. Install Ollama Linux binary
    !curl -fsSL https://ollama.com/install.sh | sh
    
    # 2. Install Python SDK
    !pip install -q ollama pandas
    
    # 3. Start the Ollama background process
    import subprocess
    import time
    print("\nStarting Ollama daemon...")
    server_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(5)  # Allow daemon to initialize

    # 4. Pull the target model (Phi-4 mini for strong reasoning)
    print("Pulling phi4-mini...")
    !ollama pull phi4-mini
    print("Ollama is ready!")

In [ ]:
import pandas as pd
import json
import random
import ollama

# Set seed for reproducibility
random.seed(42)

chunks_path = f"{base_path}/data/processed/san_diego_code_chunks.jsonl"
output_path = f"{base_path}/data/processed/evaluation_ground_truth.json"

# Read data
data = []
try:
    with open(chunks_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
    print(f"Total chunks loaded: {len(df)}")
except FileNotFoundError:
    print(f"ERROR: Could not find data at {chunks_path}. Verify the Google Drive path.")

## 1. Feature Extraction and Proportional Sampling

The evaluation focuses exclusively on regulations that influence physical development and entitlements in San Diego (Chapters 11, 13, and 14).

In [ ]:
if 'df' in locals():
    # Extract chapter metadata into a direct column
    df['chapter'] = df['metadata'].apply(lambda x: x.get('chapter', ''))

    # Filter to core development chapters
    target_chapters = ['chapter_11', 'chapter_13', 'chapter_14']
    filtered_df = df[df['chapter'].isin(target_chapters)].copy()

    # Proportional sample sizes (Target N=50)
    target_n = 50
    samples_ch14 = int(target_n * 0.50) 
    samples_ch13 = int(target_n * 0.44) 
    samples_ch11 = target_n - (samples_ch14 + samples_ch13)

    # Execute stratified sampling
    eval_set = pd.concat([
        filtered_df[filtered_df['chapter'] == 'chapter_14'].sample(n=samples_ch14, random_state=42),
        filtered_df[filtered_df['chapter'] == 'chapter_13'].sample(n=samples_ch13, random_state=42),
        filtered_df[filtered_df['chapter'] == 'chapter_11'].sample(n=samples_ch11, random_state=42)
    ]).reset_index(drop=True)

    print(f"Evaluation chunks isolated: {len(eval_set)}")

## 2. Synthetic Query Provenance via Ollama + Phi-4 Mini

Generates realistic user questions exclusively using GPU resources on the Colab instance. Phi-4 Mini provides high reasoning quality for accurately parsing the dense municipal code context.

In [ ]:
def generate_synthetic_query_ollama(chunk_text):
    prompt_text = f"""Act as a citizen or property developer in San Diego.
    Based strictly on the legal text provided below, generate ONE realistic question that a user would ask where this text is the exact, definitive answer.
    Do NOT mention section numbers or the phrase 'municipal code' in the question itself.
    Keep the question concise, practical, and conversational.
    
    Legal Text:
    {chunk_text}
    
    Question: """

    try:
        response = ollama.chat(
            model='phi4-mini',
            messages=[
                {'role': 'system', 'content': 'Act as a legal assistant specializing in the San Diego Municipal Code and Land Use regulations.'},
                {'role': 'user', 'content': prompt_text}
            ],
            options={'temperature': 0.7}
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Ollama Inference Error: {e}")
        return None

if 'eval_set' in locals():
    ground_truth_records = []
    
    print("Generating Synthetic Queries via Local Ollama (Phi-4 Mini)...\n")
    for idx, row in eval_set.iterrows():
        chunk_text = row['text']
        meta = row['metadata']
        
        # Directly generate with NO artificial rate-limiting
        synthetic_query = generate_synthetic_query_ollama(chunk_text)
        
        if synthetic_query:
            record = {
                "query_id": f"q_{idx}",
                "query": synthetic_query,
                "ground_truth_chunk_index": meta['chunk_index'],
                "ground_truth_section": meta['hierarchy'].get('section', ''),
                "chapter": row['chapter'],
                "source_text": chunk_text
            }
            ground_truth_records.append(record)
            
        if (idx + 1) % 5 == 0:
            print(f"Processed {idx + 1}/50 queries...")
    
    print(f"\nSuccessfully generated {len(ground_truth_records)} query provenances.")

## 3. Serialization and Export

The generated dataset is exported to JSON format for the evaluation script.

In [ ]:
if 'ground_truth_records' in locals() and ground_truth_records:
    # Ensure output directory exists if running in a new Colab runtime
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(ground_truth_records, f, indent=4)

    print(f"Evaluation ground truth dataset finalized and exported to: {output_path}")